# ACT 학습 (Colab)

위에서 아래로 순서대로 실행한다. 런타임이 초기화돼도 처음부터 다시 실행하면 된다 (이미 된 단계는 건너뛴다).

1. GPU 확인
2. 구글 드라이브 연결
3. LeRobot 설치
4. 데이터 준비 (드라이브의 압축 파일에서 풀기)
5. 학습: 체크포인트가 없으면 처음부터, 있으면 이어서
6. 체크포인트 확인

체크포인트는 세션(`/content`)이 아니라 **드라이브에 직접** 저장한다. 런타임이 죽어도 마지막 저장 시점까지 남는다.
본 학습으로 바꿀 때는 4번의 `DATA`, 5번의 `NAME`, `DATA`, `STEPS`, `SAVE_FREQ`를 고친다.

In [ ]:
!nvidia-smi
!python --version

In [ ]:
# 내 구글드라이브 연결 (드라이브 전체가 /content/drive/MyDrive 로 보인다)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
# lerobot 다운로드 (폴더가 있으면 건너뜀)
[ -d /content/lerobot ] || git clone -q --branch v0.6.1 --depth 1 \
  https://github.com/huggingface/lerobot /content/lerobot
cd /content/lerobot
# 설치 (이미 import 되면 건너뜀)
python -c "import lerobot" 2>/dev/null || pip install -q -e ".[training]"
# 설치 확인
python -c "import lerobot, torch; print(lerobot.__version__, torch.__version__, torch.cuda.is_available())"

In [ ]:
%%bash
# 데이터 준비: 드라이브의 압축 파일에서 풀기
# 마지막에 풀리는 파일(data/...parquet)이 있어야 건너뛴다. 반쪽만 풀린 폴더는 지우고 다시 푼다.
DATA=omx_test
: "${DATA:?}"
D=/content/drive/MyDrive/Notebook/lerobot_lab_colab
if [ ! -f /content/data/$DATA/data/chunk-000/file-000.parquet ]; then
  rm -rf /content/data/$DATA
  mkdir -p /content/data
  tar xzf $D/$DATA.tgz -C /content/data
fi
find /content/data/$DATA -type f | sort

In [ ]:
%%bash
# 학습: 드라이브에 체크포인트가 있으면 이어서, 없으면 처음부터
# - 출력 폴더(OUT)가 드라이브라서 런타임이 죽어도 SAVE_FREQ 간격까지 남는다
# - 이어 돌릴 때 STEPS는 지금까지 한 스텝보다 커야 한다 (같으면 0스텝으로 끝남)
# - 처음부터 돌리는데 OUT 폴더가 이미 있으면 에러가 난다 (NAME을 바꾸거나 폴더를 지운다)
NAME=omx_act_smoke_v2
DATA=omx_test
STEPS=500
SAVE_FREQ=250
: "${NAME:?}" "${DATA:?}"
D=/content/drive/MyDrive/Notebook/lerobot_lab_colab
OUT=$D/$NAME
cd /content/lerobot
LAST=$(ls -d $OUT/checkpoints/[0-9]* 2>/dev/null | sort | tail -1)
if [ -n "$LAST" ]; then
  echo "이어서 학습: $LAST"
  lerobot-train \
    --config_path=$LAST/pretrained_model/train_config.json \
    --resume=true --steps=$STEPS --save_freq=$SAVE_FREQ \
    --output_dir=$OUT
else
  echo "처음부터 학습: $OUT"
  lerobot-train \
    --dataset.repo_id=minodori/$DATA \
    --dataset.root=/content/data/$DATA \
    --policy.type=act --policy.device=cuda \
    --policy.push_to_hub=false \
    --batch_size=8 --steps=$STEPS --seed=1000 \
    --num_workers=2 --log_freq=50 --save_freq=$SAVE_FREQ \
    --job_name=$NAME --output_dir=$OUT
fi

In [ ]:
%%bash
# 체크포인트 확인 (위 학습 셀의 NAME과 같게)
NAME=omx_act_smoke_v2
OUT=/content/drive/MyDrive/Notebook/lerobot_lab_colab/$NAME
ls $OUT/checkpoints
du -sh $OUT